In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import shutil
import yaml
from PIL import Image

In [ ]:
RAW_DIR = Path("data/raw")
INTERIM_IMAGE_DIR = Path("data/interim/images")
INTERIM_LABEL_DIR = Path("data/interim/labels")
PROCESSED_DIR = Path("data/processed")
RFDETR_DIR = Path("data/rfdetr")

# Create output directories
for path in [
    INTERIM_IMAGE_DIR,
    INTERIM_LABEL_DIR,
    PROCESSED_DIR,
    RFDETR_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Directory setup OK")

In [34]:
image_dir = Path("data/raw/train/train")

sizes = []

for path in image_dir.glob("*"):
    try:
        with Image.open(path) as img:
            sizes.append(img.size)  # (width, height)
    except:
        pass

sizes = np.array(sizes)

mean_width, mean_height = sizes.mean(axis=0)
std_width, std_height = sizes.std(axis=0)

print(f"Number of images: {len(sizes)}")
print(f"Average resolution: {mean_width:.1f} x {mean_height:.1f}")
print(f"Std resolution:     {std_width:.1f} x {std_height:.1f}")

Number of images: 2991
Average resolution: 352.0 x 288.0
Std resolution:     0.0 x 0.0


In [35]:
IMG_W = 352
IMG_H = 288

In [36]:
df = pd.read_csv(r"data\raw\train.csv")

df

,id,image_id,class_id,x1,y1,x2,y2
0,0,1066_20260825_060110.jpg,0,152,121,186,191
1,1,1066_20260825_060110.jpg,0,119,71,144,104
2,2,1066_20260825_060110.jpg,0,114,14,123,22
3,3,1066_20260825_060110.jpg,1,212,83,223,113
4,4,1066_20260825_060110.jpg,0,48,246,101,288
...,...,...,...,...,...,...,...
27391,27391,244_20260831_204001.jpg,1,52,125,62,144
27392,27392,244_20260831_204001.jpg,0,165,54,178,83
27393,27393,244_20260831_204001.jpg,0,155,153,184,218
27394,27394,244_20260831_204001.jpg,0,0,149,23,191


In [37]:
box_cols = [
    "image_id",
    "class_id",
    "x1",
    "y1",
    "x2",
    "y2",
]

# Remove Dupe

before = len(df)

df = df.drop_duplicates(subset=box_cols).reset_index(drop=True)

print("Removed duplicates:", before - len(df))

Removed duplicates: 25


In [38]:
# Remove Invalid Box

invalid = df[
    (df["x1"] < 0) |
    (df["y1"] < 0) |
    (df["x2"] > IMG_W) |
    (df["y2"] > IMG_H) |
    (df["x2"] <= df["x1"]) |
    (df["y2"] <= df["y1"]) |
    (~df["class_id"].between(0, 7))
]

print("Invalid annotations:", len(invalid))
display(invalid.head())

Invalid annotations: 0


,id,image_id,class_id,x1,y1,x2,y2


In [39]:
df["box_width"] = df["x2"] - df["x1"]
df["box_height"] = df["y2"] - df["y1"]

print(df[["box_width", "box_height"]].describe())

          box_width    box_height
count  27371.000000  27371.000000
mean      16.889007     27.016733
std       13.234976     22.316356
min        1.000000      1.000000
25%        8.000000     12.000000
50%       13.000000     20.000000
75%       22.000000     34.000000
max      140.000000    288.000000


In [40]:
# Fix Image End Marker Corrupted

image_dir = Path(r"data\raw\train\train")

def check_img(image_dir):
    jpgs = list(image_dir.rglob("*.jpg"))

    bad = []

    for path in jpgs:
        try:
            with open(path, "rb") as f:
                f.seek(-2, 2)

                if f.read() != b"\xff\xd9":
                    bad.append(path)

        except Exception as e:
            bad.append(path)

    print("Total JPEG:", len(jpgs))
    print("Bad JPEG endings:", len(bad))
    print("Percentage:", len(bad) / len(jpgs) * 100)

check_img(image_dir)

Total JPEG: 2991
Bad JPEG endings: 2991
Percentage: 100.0


In [41]:
src_dir = Path(r"data\raw\train\train")
dst_dir = Path(r"data\interim\images")

dst_dir.mkdir(parents=True, exist_ok=True)

image_paths = list(src_dir.rglob("*.jpg"))

for src_path in tqdm(image_paths, desc="Repairing JPEGs"):
    dst_path = dst_dir / src_path.name

    try:
        with Image.open(src_path) as img:
            img = img.convert("RGB")

            img.save(
                dst_path,
                format="JPEG",
                quality=95
            )

    except Exception as e:
        print("Failed:", src_path, e)

Repairing JPEGs:   0%|          | 0/2991 [00:00<?, ?it/s]

Repairing JPEGs: 100%|██████████| 2991/2991 [00:04<00:00, 633.58it/s]


In [42]:
image_dir = Path(r"data\interim\images")
check_img(image_dir)

Total JPEG: 2991
Bad JPEG endings: 0
Percentage: 0.0


In [43]:
# Label into .txt file

label_dir = Path(r"data\interim\labels")

if label_dir.exists():
    shutil.rmtree(label_dir)

label_dir.mkdir(parents=True, exist_ok=True)

for image_name, group in tqdm(df.groupby("image_id")):

    label_path = label_dir / f"{Path(image_name).stem}.txt"

    with open(label_path, "w") as f:
        for row in group.iterrows():

            # cls_id, x, y, w, h
            x1 = row[1]["x1"]
            y1 = row[1]["y1"]
            x2 = row[1]["x2"]
            y2 = row[1]["y2"]

            x_center = ((x1 + x2) / 2) / IMG_W
            y_center = ((y1 + y2) / 2) / IMG_H

            width = (x2 - x1) / IMG_W
            height = (y2 - y1) / IMG_H

            f.write(
                f"{row[1]['class_id']} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{width:.6f} "
                f"{height:.6f}\n"
            )

100%|██████████| 2889/2889 [00:02<00:00, 1214.28it/s]


Multilabel Stratified Split

In [44]:
image_names = df["image_id"].unique()

num_classes = int(df["class_id"].max()) + 1

Y = np.zeros((len(image_names), num_classes), dtype= int)

image_to_idx = {image_name: i for i, image_name in enumerate(image_names)}

for image_name, group in df.groupby("image_id"):
    idx = image_to_idx[image_name]

    classes = group["class_id"].unique()

    for cls_id in classes:
        Y[idx, int(cls_id)] = 1

Y

array([[1, 1, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 0],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 1, 0],
       [1, 1, 0, ..., 0, 0, 0]], shape=(2889, 8))

In [45]:
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(splitter.split(image_names, Y))

train_images = image_names[train_idx]
val_images = image_names[val_idx]

In [46]:
print("Train images:", len(train_images))
print("Val images:", len(val_images))

Train images: 2310
Val images: 579


YOLO Structure

In [47]:
src_img_dir = Path(r"data\interim\images")
src_lbl_dir = Path(r"data\interim\labels")

dst = Path(r"data\processed")

if dst.exists():
    shutil.rmtree(dst)

for split in ["train", "val"]:
    (dst / "images" / split).mkdir(parents=True, exist_ok=True)
    (dst / "labels" / split).mkdir(parents=True, exist_ok=True)

In [48]:
def copy_split(image_names, split):
    for image_name in tqdm(image_names, desc=f"Copying {split}"):
        image_name = Path(image_name)

        src_img = src_img_dir / image_name
        src_lbl = src_lbl_dir / f"{image_name.stem}.txt"

        dst_img = dst / "images" / split / image_name.name
        dst_lbl = dst / "labels" / split / f"{image_name.stem}.txt"

        shutil.copy2(src_img, dst_img)
        shutil.copy2(src_lbl, dst_lbl)


copy_split(train_images, "train")
copy_split(val_images, "val")

Copying val: 100%|██████████| 579/579 [00:04<00:00, 141.34it/s]


In [49]:
class_names = [
    "Car",
    "Motorcycle",
    "Bus",
    "Truck",
    "Tuktuk",
    "Van",
    "Pickup",
    "Songthaew",
]

config = {
    "path": str(dst.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {
        i: name
        for i, name in enumerate(class_names)
    }
}

with open(dst / "data.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

DETR Strcuture

In [50]:
src = Path(r"data/processed")
dst = Path(r"data/rfdetr")

# Start clean
if dst.exists():
    shutil.rmtree(dst)

# Create RF-DETR structure
for split in ["train", "valid"]:
    (dst / split / "images").mkdir(parents=True, exist_ok=True)
    (dst / split / "labels").mkdir(parents=True, exist_ok=True)

# Copy train
for p in (src / "images" / "train").iterdir():
    if p.is_file():
        shutil.copy2(p, dst / "train" / "images" / p.name)

for p in (src / "labels" / "train").glob("*.txt"):
    shutil.copy2(p, dst / "train" / "labels" / p.name)

# Copy val -> valid
for p in (src / "images" / "val").iterdir():
    if p.is_file():
        shutil.copy2(p, dst / "valid" / "images" / p.name)

for p in (src / "labels" / "val").glob("*.txt"):
    shutil.copy2(p, dst / "valid" / "labels" / p.name)

print("Done")

Done


In [ ]:
config = {
    "train": "train/images",
    "val": "valid/images",
    "nc": 8,
    "names": {
            i: name
            for i, name in enumerate(class_names)
            }
}

with open(dst / "data.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Done")

Done
